# NIDRA — Full-Scale Training on Colab (GPU side-by-side run)

This notebook runs the same `config/default.yaml` training pipeline as the
project's local `PRODUCTION_RUN_GUIDE.md`, on a Colab GPU, so you can
compare wall-clock time against a CPU-only laptop run.

**How to use this notebook**: `Runtime -> Change runtime type -> GPU (T4)`,
then `Runtime -> Run all`. Every cell is written to run unattended after
that, with **one unavoidable manual step**: the Google Drive authorization
popup in the "Mount Google Drive" cell below (Google requires a human click
there; nothing else does).

## One-time prerequisite (do this before clicking Run all)

This project's dataset (CIC-IDS2017 flow CSVs + this project's own
tshark-extracted packet parquet files) is **not** in the git repo — it's
gitignored (multi-GB) and can't be quickly re-downloaded or re-extracted on
a fresh Colab VM. You need to upload it to Google Drive **once**, and this
notebook mounts your Drive and reads it from there every time you open it.

Upload these two folders to your Google Drive, preserving this exact
structure, under `MyDrive/nidra_data/cicids2017/`:

```
MyDrive/nidra_data/cicids2017/
├── csv/extracted/TrafficLabelling /        <- the 8 labelled flow CSVs
│   ├── Monday-WorkingHours.pcap_ISCX.csv
│   ├── Tuesday-WorkingHours.pcap_ISCX.csv
│   ├── Wednesday-workingHours.pcap_ISCX.csv
│   ├── Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
│   ├── Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
│   ├── Friday-WorkingHours-Morning.pcap_ISCX.csv
│   ├── Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
│   └── Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
└── pcap/parquet/                            <- the 5 already-extracted packet parquets
    ├── Monday-WorkingHours_packets.parquet
    ├── Tuesday-WorkingHours_packets.parquet
    ├── Wednesday-workingHours_packets.parquet
    ├── Thursday-WorkingHours_packets.parquet
    └── Friday-WorkingHours_packets.parquet
```

Note the trailing space in `TrafficLabelling ` — that's the real name of the
official CIC-IDS2017 distribution's folder, not a typo. You do **not** need
to upload raw `.pcap` files or the `MachineLearningCVE` CSV variant —
neither is read by this pipeline. Total upload is roughly 1.5-2GB, far
smaller than the raw PCAPs (which the parquet files were already extracted
from on your laptop).

If your Drive path differs from `MyDrive/nidra_data/cicids2017`, edit
`DRIVE_DATA_DIR` in the "Locate your data on Drive" cell below.


In [ ]:
#@title 1. Check GPU and system RAM
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or
      "nvidia-smi not found — no GPU attached.")

try:
    import torch
except ImportError:
    torch = None

if torch is not None and torch.cuda.is_available():
    print(f"torch sees a GPU: {torch.cuda.get_device_name(0)}")
    DEVICE = "cuda"
else:
    print("No CUDA GPU visible to torch. Go to Runtime -> Change runtime "
          "type -> Hardware accelerator -> GPU (T4), then Runtime -> "
          "Restart and run all. Continuing on CPU for now, but that "
          "defeats the point of this notebook.")
    DEVICE = "cpu"

with open("/proc/meminfo") as f:
    total_kb = int([l for l in f if l.startswith("MemTotal")][0].split()[1])
TOTAL_RAM_GB = total_kb / 1e6
print(f"Total system RAM: {TOTAL_RAM_GB:.1f} GB")

# The Mac reference run used --max-train-samples 500000 --max-val-samples
# 50000 and measured ~8.7GB peak RSS. Colab's free-tier default runtime
# gives ~12-13GB RAM, which is tighter than the 16GB Mac this was tuned on.
# Scale the cap down automatically on a small-RAM runtime rather than risk
# an OOM kill; scale it up to match the Mac exactly if you got a High-RAM
# runtime (Colab Pro, or a lucky free allocation).
if TOTAL_RAM_GB >= 20:
    MAX_TRAIN_SAMPLES, MAX_VAL_SAMPLES = 500_000, 50_000
elif TOTAL_RAM_GB >= 15:
    MAX_TRAIN_SAMPLES, MAX_VAL_SAMPLES = 400_000, 40_000
else:
    MAX_TRAIN_SAMPLES, MAX_VAL_SAMPLES = 250_000, 25_000
print(f"Using --max-train-samples {MAX_TRAIN_SAMPLES} "
      f"--max-val-samples {MAX_VAL_SAMPLES} for this runtime "
      f"({TOTAL_RAM_GB:.1f} GB RAM detected).")
print("If this differs from your laptop's cap, the timing comparison below "
      "reports samples/sec as well as seconds/epoch so the two runs stay "
      "comparable even at different cap sizes.")


In [ ]:
#@title 2. Mount Google Drive (the one manual step — click through the auth popup)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title 3. Locate your data on Drive and verify it's complete
import os
from pathlib import Path

DRIVE_DATA_DIR = "/content/drive/MyDrive/nidra_data/cicids2017"  #@param {type:"string"}

flow_dir = Path(DRIVE_DATA_DIR) / "csv" / "extracted" / "TrafficLabelling "
packets_dir = Path(DRIVE_DATA_DIR) / "pcap" / "parquet"

expected_csvs = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
]
expected_parquets = [
    "Monday-WorkingHours_packets.parquet",
    "Tuesday-WorkingHours_packets.parquet",
    "Wednesday-workingHours_packets.parquet",
    "Thursday-WorkingHours_packets.parquet",
    "Friday-WorkingHours_packets.parquet",
]

missing = []
print(f"Checking {flow_dir} ...")
for name in expected_csvs:
    ok = (flow_dir / name).exists()
    print(f"  {'OK ' if ok else 'MISSING'}  {name}")
    if not ok:
        missing.append(str(flow_dir / name))

print(f"\nChecking {packets_dir} ...")
for name in expected_parquets:
    ok = (packets_dir / name).exists()
    print(f"  {'OK ' if ok else 'MISSING'}  {name}")
    if not ok:
        missing.append(str(packets_dir / name))

if missing:
    raise FileNotFoundError(
        "Data upload is incomplete — see the prerequisite section at the "
        "top of this notebook for the exact folder structure needed.\n"
        "Missing:\n" + "\n".join(missing)
    )
print("\nAll expected files found.")


In [ ]:
#@title 4. Clone the repo (public GitHub repo, MLimplement branch)
import os

REPO_DIR = "/content/nidra"
BRANCH = "MLimplement"

if os.path.isdir(REPO_DIR):
    print("Repo already present, pulling latest instead of re-cloning.")
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} https://github.com/clustercoder/nidra.git {REPO_DIR}

!cd {REPO_DIR} && git log --oneline -3


In [ ]:
#@title 5. Install the nidra package (editable install)
%pip install -q -e /content/nidra/ml
import importlib
import nidra
importlib.reload(nidra)
print("nidra package importable OK")


In [ ]:
#@title 6. Symlink the Drive data into the path the config expects (~/nidra/cicids2017)
import os
from pathlib import Path

home = Path.home()
target = home / "nidra" / "cicids2017"
target.parent.mkdir(parents=True, exist_ok=True)

if target.is_symlink() or target.exists():
    if target.is_symlink():
        target.unlink()
    else:
        raise RuntimeError(
            f"{target} exists and is a real directory, not a symlink — "
            "remove or rename it manually before re-running this cell."
        )

os.symlink(DRIVE_DATA_DIR, target)
print(f"Symlinked {target} -> {DRIVE_DATA_DIR}")
assert (target / "csv" / "extracted" / "TrafficLabelling " / "Monday-WorkingHours.pcap_ISCX.csv").exists()
print("Symlink verified working.")


In [ ]:
#@title 7. Sanity check: run the test suite before spending GPU time
%cd /content/nidra/ml
!python -m pytest tests/ -q


In [ ]:
#@title 8. Timing probe — one epoch, same sample cap on both machines' *rate*
import subprocess
import time

%cd /content/nidra/ml

cmd = [
    "python", "-m", "nidra.train.train_dynamics",
    "--config", "config/default.yaml",
    "--epochs", "1", "--seed", "0",
    "--max-train-samples", str(MAX_TRAIN_SAMPLES),
    "--max-val-samples", str(MAX_VAL_SAMPLES),
    "--device", DEVICE,
]
print("Running:", " ".join(cmd))
t0 = time.time()
result = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.time() - t0
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError("Timing probe failed — see stderr above before continuing.")

samples_per_sec = MAX_TRAIN_SAMPLES / elapsed
print(f"\nTotal wall time (incl. data loading + scaler fit): {elapsed:.1f}s")
print(f"Effective throughput: {samples_per_sec:.0f} train samples/sec")
print(f"Projected Stage 1 total (60 epochs x 5 seeds), "
      f"data-loading cost paid once: "
      f"~{(elapsed * 60 * 5) / 3600:.1f} hours if repeated naively per seed, "
      f"or less if you use the single-process 5-seed run below (data is "
      f"loaded once, not per seed).")
print("\nCompare against the reference CPU (16GB Apple M1) measurement: "
      "~166-176s/epoch at 500,000 train samples "
      "(~2,850-3,000 samples/sec).")


## Decision point

Look at the throughput number printed above. If it's meaningfully higher
than the ~2,850-3,000 samples/sec CPU reference, the GPU run below is worth
finishing. If it's about the same or slower (small models like this one
don't always benefit much from a GPU — the bottleneck can be Python/data
loading overhead rather than matrix math), it's not worth burning Colab
compute quota on the full run; just keep the laptop run going instead.

Everything below this point is the real production run: Stage 1 (60
epochs), Stage 2 (30 epochs), each across all 5 ensemble seeds, then
calibration + evaluation + benchmarking + report generation — the same
steps as `PRODUCTION_RUN_GUIDE.md`, just running here instead of locally.


In [ ]:
#@title 9. Full training run — one seed at a time, synced to Drive after each
# Colab (especially the free tier) can disconnect mid-run, and this
# pipeline only saves a checkpoint at the END of each seed's training (see
# nidra/train/train_dynamics.py::train_one_seed) — so unlike the
# single-process "train all 5 seeds at once" command recommended for a
# stable local machine, here we run ONE SEED AT A TIME as a separate
# process and copy its output to Drive immediately. A disconnect then only
# costs you the seed in progress, not everything.
import os
import subprocess
import time
import shutil
from pathlib import Path

%cd /content/nidra/ml

DRIVE_RUN_DIR = "/content/drive/MyDrive/nidra_data/colab_run_artifacts"
os.makedirs(DRIVE_RUN_DIR, exist_ok=True)

SEEDS = [0, 1, 2, 3, 4]
overall_start = time.time()

def run(cmd, label):
    print(f"\n===== {label}: {' '.join(cmd)} =====")
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    dt = time.time() - t0
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print(proc.stderr[-4000:])
        print(f"!!! {label} FAILED after {dt/60:.1f} min (exit {proc.returncode}) — "
              f"continuing to the next seed rather than aborting the whole loop.")
        return False, dt
    print(f"{label} finished in {dt/60:.1f} min")
    return True, dt

for seed in SEEDS:
    ok1, _ = run(
        ["python", "-m", "nidra.train.train_dynamics", "--config", "config/default.yaml",
         "--seed", str(seed),
         "--max-train-samples", str(MAX_TRAIN_SAMPLES), "--max-val-samples", str(MAX_VAL_SAMPLES),
         "--device", DEVICE],
        f"Stage 1 (dynamics), seed={seed}",
    )
    if not ok1:
        continue
    ok2, _ = run(
        ["python", "-m", "nidra.train.train_heads", "--config", "config/default.yaml",
         "--seed", str(seed),
         "--max-train-samples", str(MAX_TRAIN_SAMPLES), "--max-val-samples", str(MAX_VAL_SAMPLES),
         "--device", DEVICE],
        f"Stage 2 (heads), seed={seed}",
    )
    # Sync whatever we have to Drive after every seed, whether or not it
    # fully succeeded — partial progress is still worth keeping.
    print(f"Syncing artifacts/ to Drive after seed={seed} ...")
    shutil.copytree("artifacts", DRIVE_RUN_DIR + "/artifacts", dirs_exist_ok=True)
    print("Synced.")

print(f"\nAll seeds attempted. Total elapsed: {(time.time() - overall_start)/3600:.2f} hours")


In [ ]:
#@title 10. Fit calibration on the newly-trained ensemble
%cd /content/nidra/ml
!python -m nidra.scripts.fit_calibration --config config/default.yaml


In [ ]:
#@title 11. Evaluate on test + holdout splits
%cd /content/nidra/ml
!python -m nidra.eval.run_eval --config config/default.yaml --seed 0 --split test    --n-samples 50 --max-eval-samples 4000
!python -m nidra.eval.run_eval --config config/default.yaml --seed 0 --split holdout --n-samples 50 --max-eval-samples 4000


In [ ]:
#@title 12. Benchmark serving latency
%cd /content/nidra/ml
!python -m nidra.serve.benchmark --weights-dir artifacts/weights --scaler-path artifacts/scaler/robust_scaler.joblib --config config/default.yaml


In [ ]:
#@title 13. Generate report artifacts (plots, metadata)
%cd /content/nidra/ml
!python -m nidra.scripts.generate_report --config config/default.yaml \
    --test-metrics-dir artifacts/metrics/test \
    --holdout-metrics-dir artifacts/metrics/holdout \
    --reports-dir reports \
    --metadata-dir artifacts/metadata


In [ ]:
#@title 14. Persist everything to Drive and print a final summary
import json
import shutil
import time
from pathlib import Path

%cd /content/nidra/ml

DRIVE_RUN_DIR = "/content/drive/MyDrive/nidra_data/colab_run_artifacts"
timestamp = time.strftime("%Y%m%d_%H%M%S")
final_dir = f"{DRIVE_RUN_DIR}_{timestamp}"

for sub in ["artifacts", "reports"]:
    if Path(sub).exists():
        shutil.copytree(sub, f"{final_dir}/{sub}", dirs_exist_ok=True)
print(f"Copied artifacts/ and reports/ to Drive at: {final_dir}")
print("Download this folder (or point a future Colab run's DRIVE_DATA_DIR-style "
      "variable at it) before this runtime recycles — Colab VMs are ephemeral.")

def show_metric(split):
    path = Path(f"artifacts/metrics/{split}/baselines.json")
    if not path.exists():
        print(f"{split}: no baselines.json found (evaluation step may have failed)")
        return
    data = json.loads(path.read_text())
    wm = data.get("world_model", {})
    persistence = data.get("persistence", {})
    print(f"{split}: world_model auc_pr={wm.get('auc_pr')}  "
          f"persistence auc_pr={persistence.get('auc_pr')}  "
          f"(world model should score higher — see MODEL_CARD.md)")

print("\n=== Final results summary ===")
show_metric("test")
show_metric("holdout")
